# OCONUS NGWPC Hydrofabric Demo
This notebook walks through the following PI-9 acceptance criteria for the OCONUS (Alaska, Hawaii, Puerto Rico/Virgin Islands) domains. This should be run after building OCONUS NHF.

- domains include PRVI, Hawaii, and Alaska
- existence of the same river miles covered in the operational version of the NWM
- ensuring rivers can be represented as a directed acyclic graphs for routing
- connectivity checks
- flowpath and divide statistics (drainage area, length, etc)
- Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.
- POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses.
- Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

In [ ]:
import json
import pprint
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [ ]:
data_dir = Path("../data")
VERSION = "1.1.4"

path_ak = data_dir / f"ak_nhf_{VERSION}.gpkg"
path_hi = data_dir / f"hi_nhf_{VERSION}.gpkg"
path_prvi =  data_dir / f"prvi_nhf_{VERSION}.gpkg"
path_nhf = data_dir / f"nhf_{VERSION}.gpkg"
path_nwm_v3 = data_dir / "NWM_v3_hydrofabric.gdb"

path_ak_val = data_dir / f"ak_nhf_{VERSION}_validation.json"
path_hi_val = data_dir / f"hi_nhf_{VERSION}_validation.json"
path_prvi_val = data_dir / f"prvi_nhf_{VERSION}_validation.json"

km_to_miles = 0.6213712

crs = {"AK":3338, "PRVI":6566, "HI":32604, "CONUS":5070}

pd.set_option("display.max_columns", None)

## Existence of the same river miles covered in the operational version of the NWM

Virtual flowpaths is the routing layer that includes all flowpaths. The sum of virtual flowpath is the sum of all NHF river miles.

In [ ]:
gdf_ak_vfp = gpd.read_file(path_ak, layer="virtual_flowpaths")
gdf_hi_vfp = gpd.read_file(path_hi, layer="virtual_flowpaths")
gdf_prvi_vfp = gpd.read_file(path_prvi, layer="virtual_flowpaths")
gdf_sconus_vfp = gpd.read_file(path_nhf, layer="virtual_flowpaths")

gdf_ak_v3 = gpd.read_file(path_nwm_v3, layer="nwm_reaches_alaska")
gdf_prvi_v3 = gpd.read_file(path_nwm_v3, layer="nwm_reaches_puertorico")
gdf_hi_v3 = gpd.read_file(path_nwm_v3, layer="nwm_reaches_hawaii")
gdf_conus_v3 = gpd.read_file(path_nwm_v3, layer="nwm_reaches_conus")
gdf_ak_v3 = gpd.read_file(path_nwm_v3, layer="nwm_reaches_alaska")

In [ ]:
# NHF in miles
domains = ["AK", "HI", "PRVI", "CONUS"]

sum_nhf = 0
for gdf in [gdf_ak_vfp, gdf_hi_vfp, gdf_prvi_vfp, gdf_sconus_vfp]:
    sum_nhf += int(gdf["geometry"].length.sum() / 1000 * km_to_miles)

sum_nwm = 0
for domain, gdf in zip(domains, [gdf_ak_v3, gdf_hi_v3 , gdf_prvi_v3, gdf_conus_v3 ]):
    # NWM v3 in miles after converting to NHF meters CRS. NWM is in 4269 / degrees
    gdf = gdf.to_crs(crs[domain])
    sum_nwm += int(gdf["geometry"].length.sum() / 1000 * km_to_miles)

print(f"Sum NHF River Miles: {sum_nhf}")
print(f"Sum NWM v3 River Miles: {sum_nwm}")


## Ensuring rivers can be represented as a directed acyclic graphs for routing / connectivity checks

The following code builds a graph from NHF and tests with the networkx `is_directed_acyclic_graph` function. 

In [ ]:
"""Check that the NHF nexus-mediated graph is a DAG."""

import sqlite3

import networkx as nx


def check_graph(GPKG: Path, domain: str) -> None:
    con = sqlite3.connect(GPKG)

    total_flowpaths = con.execute("SELECT count(*) FROM flowpaths").fetchone()[0]
    total_nexuses = con.execute("SELECT count(*) FROM nexus").fetchone()[0]

    fp_to_nex = con.execute("SELECT fp_id, dn_nex_id FROM flowpaths").fetchall()
    nex_to_fp = con.execute("SELECT nex_id, dn_fp_id FROM nexus WHERE dn_fp_id IS NOT NULL").fetchall()

    con.close()

    G = nx.DiGraph()

    fp_ids = set()
    nex_ids = set()

    # IMPORTANT: The fp_to_nex edge construction adds all outlet flowpaths because all flowpaths (including outlet flowpaths) have a dn_nex_id
    for fp, nex in fp_to_nex:
        fp_node = f"fp_{fp}"
        nex_node = f"nex_{nex}"
        G.add_edge(fp_node, nex_node)
        fp_ids.add(fp_node)
        nex_ids.add(nex_node)

    # IMPORTANT: The 12,035 outlet nexuses that are excluded from nex_to_fp are included in fp_to_nex and so they still get included in the graph in the loop above.
    for nex, fp in nex_to_fp:
        nex_node = f"nex_{nex}"
        fp_node = f"fp_{fp}"
        G.add_edge(nex_node, fp_node)
        nex_ids.add(nex_node)
        fp_ids.add(fp_node)

    print(f"{domain}")
    print(f"Total flowpaths in NHF {domain}:    {total_flowpaths:,}")
    print(f"Total nexuses in NHF {domain}:      {total_nexuses:,}")
    print(f"Flowpath nodes in graph {domain}:   {len(fp_ids):,}")
    print(f"Nexus nodes in graph {domain}:      {len(nex_ids):,}")
    print(f"Total graph nodes {domain}:         {G.number_of_nodes():,}")
    print(f"Total graph edges {domain}:         {G.number_of_edges():,}")
    print("")

    is_dag = nx.is_directed_acyclic_graph(G)
    print(f"{domain} Is DAG: {is_dag}")
    print("")

    if not is_dag:
        cycles = list(nx.simple_cycles(G))
        print(f"Found {len(cycles):,} cycle(s)")
        for c in cycles[:5]:
            print(f"  {c}")


In [ ]:
check_graph(path_ak, "AK")
check_graph(path_hi, "HI")
check_graph(path_prvi, "PRVI")

In [ ]:
"""Visualize NHF nexus-mediated graph structure for a subset of flowpaths."""

from collections import defaultdict, deque

import matplotlib.pyplot as plt


def visualize_nexus_graph(gpkg: Path, outlet_fp_id: int, max_nodes: int, fig_path: Path, domain: str):
    con = sqlite3.connect(gpkg)

    fp_to_nex_all = con.execute("SELECT fp_id, dn_nex_id FROM flowpaths").fetchall()
    nex_to_fp_all = con.execute(
        "SELECT nex_id, dn_fp_id FROM nexus WHERE dn_fp_id IS NOT NULL"
    ).fetchall()

    con.close()

    fp_to_fp_all = []
    nex_dn = {nex: fp for nex, fp in nex_to_fp_all}
    for fp, nex in fp_to_nex_all:
        dn_fp = nex_dn.get(nex)
        if dn_fp is not None:
            fp_to_fp_all.append((fp, dn_fp))

    upstream = defaultdict(list)
    for fp, to_fp in fp_to_fp_all:
        upstream[int(to_fp)].append(int(fp))

    fp_subset = set()
    queue = deque([outlet_fp_id])
    while queue and len(fp_subset) < max_nodes:
        node = queue.popleft()
        if node not in fp_subset:
            fp_subset.add(node)
            queue.extend(upstream.get(node, []))

    fp_nex_map = {fp: nex for fp, nex in fp_to_nex_all if fp in fp_subset}
    nex_subset = set(fp_nex_map.values())

    fp_to_nex_sub = [(fp, nex) for fp, nex in fp_to_nex_all if fp in fp_subset]
    nex_to_fp_sub = [(nex, fp) for nex, fp in nex_to_fp_all if nex in nex_subset and fp in fp_subset]

    print(f"{len(fp_subset)} flowpaths, {len(nex_subset)} nexus nodes")

    FP_COLOR = "#4A90D9"
    NEX_COLOR = "#E8783A"

    G = nx.DiGraph()
    for fp in fp_subset:
        G.add_node(f"fp_{fp}", ntype="fp")
    for nex in nex_subset:
        G.add_node(f"nex_{nex}", ntype="nex")
    fp_nex_edges = [(f"fp_{fp}", f"nex_{nex}") for fp, nex in fp_to_nex_sub]
    nex_fp_edges = [(f"nex_{nex}", f"fp_{fp}") for nex, fp in nex_to_fp_sub]
    G.add_edges_from(fp_nex_edges)
    G.add_edges_from(nex_fp_edges)

    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
    except Exception:
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

    fp_nodes = [n for n in G if G.nodes[n].get("ntype") == "fp"]
    nex_nodes = [n for n in G if G.nodes[n].get("ntype") == "nex"]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(
        f"{domain} NHF nexus-mediated graph ({len(fp_subset)} flowpaths near fp_id={outlet_fp_id})",
        fontsize=12, fontweight="bold",
    )

    nx.draw_networkx_nodes(G, pos, nodelist=fp_nodes, node_color=FP_COLOR, node_size=40, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=nex_nodes, node_color=NEX_COLOR, node_size=25, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=fp_nex_edges, ax=ax, edge_color=FP_COLOR,
                        arrows=True, arrowsize=6, width=1.0, alpha=0.5)
    nx.draw_networkx_edges(G, pos, edgelist=nex_fp_edges, ax=ax, edge_color="#D62728",
                        arrows=True, arrowsize=10, width=2.0, style="dashed",
                        connectionstyle="arc3,rad=0.3")

    ax.legend(handles=[
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=FP_COLOR, markersize=8, label=f"flowpath ({len(fp_nodes)})"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=NEX_COLOR, markersize=8, label=f"nexus ({len(nex_nodes)})"),
        plt.Line2D([0], [0], color=FP_COLOR, linewidth=1, label="fp → nex"),
        plt.Line2D([0], [0], color="#D62728", linewidth=1.5, linestyle="dashed", label="nex → fp"),
    ], loc="upper left", fontsize=8)
    ax.axis("off")

    fig.tight_layout()
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")


In [ ]:
MAX_NODES = 120

print("AK")
visualize_nexus_graph(path_ak, 118077, MAX_NODES, "ak_graph.png", "AK")

print("HI")
visualize_nexus_graph(path_hi, 770, MAX_NODES, "hi_graph.png", "HI")

print("PRVI")
visualize_nexus_graph(path_prvi, 21897, MAX_NODES, "prvi_graph.png", "PRVI")

## Flowpath and divide statistics (drainage area, length, etc)
Show the attributes available for divides and flowpath layers.

In [ ]:
def show_attributes(domain_path: Path, domain_name: str):
    """Display the fields in flowpaths and divides"""
    gdf_fp = gpd.read_file(domain_path, layer="flowpaths")
    print(f"{domain_name} Flowpath attributes")
    display(pd.DataFrame(data={"Columns":gdf_fp.columns}))

    gdf_div = gpd.read_file(domain_path, layer="divides")
    print(f"{domain_name} Divide Attributes")
    display(pd.DataFrame(data={"Columns":gdf_div.columns}))

In [ ]:
# AK
show_attributes(path_ak, domain_name="AK")

In [ ]:
# HI
show_attributes(path_hi, domain_name="HI")

In [ ]:
# PRVI
show_attributes(path_hi, domain_name="PRVI")

## Ensure that hydrofabric fully complies with the HY_Features (WaterML2 Part 3) data model standard as extended to include flowlines that reduce artificial trans-basin transfers of flow.

oCONUS NHF follows the same schema as Super CONUS NHF which fully compiles with HY_Features. The schema can be viewed in the OE at: https://edfs.oe.nextgenwaterprediction.com/dashboard/hydrofabric_dash > click "Data Model/Schemas". 

Since NHF v1.0.0, additional attributes (e.g fp_id, virtual_fp_id, dn_nex_id) have been added to some tables (e.g. gages, lakes) to reduce required table joins. 

## POIs should include at minimum all existing operational data assimilation gages as well as existing calibration gages and waterbodies (i.e. lake/reservoir) and their outlet nexuses

### Waterbodies / Lakes
The `lakes` layer was built in NHF to be a 1:1 representation of NWM operational waterbodies. The `lakes` layer retains all data from Hydrofabric 2.2 (Puerto Rico and Hawaii) and LAKEPARM (Alaska).

Where polygons were available (HI and PRVI), lakes were mapped to the most downstream intersecting flowpath. The most downstream flowpath is chosen with the minimum hydrosequence. In Alaska, points were mapped to their nearest flowpath.

In [ ]:
def compare_lakes(nhf_path: Path, nwm_path: Path, domain: str, id_field: str):
    """Compare if lakes are present in an NWM source file and an NHF gages layer"""
    gdf_nwm = gpd.read_file(nwm_path)
    gdf_nhf = gpd.read_file(nhf_path, layer="lakes")
    print(f"{domain} NHF lakes: {len(gdf_nhf)}")
    print(f"{domain} NWM lakes: {len(gdf_nwm)}")
    print(f"{domain} lakes COMID in NWM: {len(gdf_nhf.loc[gdf_nhf['lake_id'].isin(gdf_nwm[id_field])])}")
    display(gdf_nhf.head())

Alaska lakes were retrieved from [NWM v3.0.18 LAKEPARM_AK.nc](https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.18/parm/domain_alaska/LAKEPARM_AK.nc) and saved to a GPKG.

In [ ]:
compare_lakes(path_ak, data_dir / "lakes/input/ak_lakeparm.gpkg", "AK", "lake_id" )
with open(path_ak_val) as f:
    pprint(json.load(f)["Lakes"])

Hawaii lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Hawaii state borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_hi, data_dir / "lakes/input/nwm_lakes_hi_input.gpkg", "HI", "newID")
with open(path_hi_val) as f:
    pprint(json.load(f)["Lakes"])

Puerto Rico / Virgin Islands lakes were extracted from `nwm_lakes.gpkg` provided by OWP in January 2026. The file was clipped to Puerto Rico/Virgin Islands borders. `lake_id` and `newID` are both NHD COMID.

In [ ]:
compare_lakes(path_prvi, data_dir / "lakes/input/nwm_lakes_prvi_input.gpkg", "PRVI", "newID")
with open(path_prvi_val) as f:
    pprint(json.load(f)["Lakes"])

### Gages
Gages were extracted from routelink and USGS. Routelink files were downloaded from NWM v3.0.18, converted to GPKG in EPSG:4326, and extracted any row with a populated gage ID field. 

If upstream area information was available, gages were matched to flowpaths/divides using it. If it was not available, gages were matched to nearest flowpath. See connectivity columns in tables below (fp_id, virtual_fp_id, dn_nex_id, dn_virtual_nex_id).

Compare gages function shows the number of gages in NHF and routelink. The validation file describes additional information about gages.

In [ ]:
def compare_gages(nhf_path: Path, routelink_path: Path, domain: str):
    """Compare if gages are present in routelink and an NHF gages layer"""
    gdf_routelink = gpd.read_file(routelink_path)

    # extract rows with gages
    gdf_routelink["gages"] = gdf_routelink["gages"].str.strip()
    gdf_routelink = gdf_routelink.loc[gdf_routelink["gages"] != ""].copy()

    gdf_gages = gpd.read_file(nhf_path, layer="gages")
    print(f"{domain} NHF gages: {len(gdf_gages)}")
    print(f"{domain} Routelink gages: {len(gdf_routelink)}")
    print(f"{domain} gage ID in Routelink: {len(gdf_gages.loc[gdf_gages['site_no'].isin(gdf_routelink['gages'])])}")
    display(gdf_gages.head())

In [ ]:
# Function used in NHF-builds to extract routelink gages - this is for demonstration purposes only
def append_from_routelink(
    gdf: gpd.GeoDataFrame, routelink: Path, id_col_name: str, shape: Path | None
) -> gpd.GeoDataFrame:
    """Append gages from RouteLink file to GeoDataFrame

    Use ogr2ogr to convert NC file to GPKG and add EPSG:4326 georef i.e. ogr2ogr RouteLink.gpkg RouteLink.nc -t_srs EPSG:4326 -s_srs EPSG:4326

    Parameters
    ----------
    gdf: GeoDataFrame
        Input dataframe to append to
    routelink : Path
        RouteLink file to extract from
    id_col_name: str
        Column to pull from for site_no in RouteLink
    shape: Path | None
        Shapefile to use for clipping
    """
    gages = gpd.read_file(routelink).to_crs(gdf.crs)

    # first get gages only
    gages = gages.loc[gages[id_col_name].str.strip() != ""].copy()

    # then check intersection if requested
    if shape:
        # Get boundary to clip to
        shp = gpd.read_file(shape).to_crs(gdf.crs)
        merged_geom = shp["geometry"].union_all()
        gages = gages.loc[gages["geometry"].intersects(merged_geom), :].copy()

    gages = gages.rename(columns={id_col_name: "site_no"})
    gages["site_no"] = gages["site_no"].str.strip()

    gages = gpd.GeoDataFrame(gages[["geometry", "site_no"]][~gages["site_no"].isin(gdf["site_no"])].copy())
    # logger.info(f"gages: added {len(gages)} gages from RouteLink not already present in dataset") # commetned for missing imports in demonstration
    gages["status"] = "routelink"
    gages = pd.concat([gdf, gages])
    gages["geometry"] = gages["geometry"].force_2d()

    return gages

In [ ]:
# AK - Expected: 2 coastal routelink gages are missing
compare_gages(path_ak, Path("../data/gages/routelink/RouteLink_AK_EPSG4326.gpkg"), "AK")

with open(path_ak_val) as f:
    pprint(json.load(f)["Gages"])

In [ ]:
# HI
compare_gages(path_hi, Path("../data/gages/routelink/RouteLink_HI_EPSG4326.gpkg"), "Hawaii")

with open(path_hi_val) as f:
    pprint(json.load(f)["Gages"])

In [ ]:
# PRVI
compare_gages(path_prvi, Path("../data/gages/routelink/RouteLink_PRVI_EPSG4326.gpkg"), "PRVI")

with open(path_prvi_val) as f:
    pprint(json.load(f)["Gages"])

## Gage / Lake Flowpath Association
In the following example, a lake COMID and USGS gage are plotted with their associated downstream nexus. 

In [ ]:
gdf_lk = gpd.read_file(path_prvi, layer="lakes")
gdf_fp = gpd.read_file(path_prvi, layer="flowpaths")
gdf_nex = gpd.read_file(path_prvi, layer="nexus")
gdf_gage = gpd.read_file(path_prvi, layer="gages")
gdf_lk_poly = gpd.read_file("../data/lakes/input/nwm_lakes_prvi_input.gpkg")

Below, a map will display a series of NHF information in Puerto Rico.

- A lake point (COMID 800043415) is plotted in red. The point geometry is in the centroid of the lake polygon. This centroid is not in the lake geometry.
- A lake polygon (COMID 800043415) is plotted in bright blue. It follows the outline of the basemap lake.
- A gage point (USGS site no 50027200) is plotted in green to the west of a dam.
- All NHF flowpaths are plotted in navy blue.
- All NHF nexus are plotted in mageneta.
- The downstream nexus that gage and lakes are mapped to circles a magenta nexus in orange.

Although the lake point does not intersect the polygon, it is mapped to the most downstream flowpath intersecting the polygon. 

The gage is mapped to the nearest flowpath. If upstream gage area is available, gages can be mapped to the most similar upstream area nexus.

In this case, both lake and gage outlets are correctly mapped to the outlet of the lake.

Because the lakes are mapped to nexus using the lake polygon, identifying lake point nexuses may show associated nexus far from the point geometry itself.

In [ ]:
comid = 800043415
gage = "50027200"

lk_pt = gdf_lk.loc[gdf_lk["lake_id"] == comid, :].copy()
lk_poly = gdf_lk_poly.loc[gdf_lk_poly["newID"] == comid, :].copy()
gage_pt = gdf_gage.loc[gdf_gage["site_no"] == gage, :].copy()

nex_id_lk = lk_pt["dn_nex_id"].values
nex_id_gage = gage_pt["dn_nex_id"].values

print(f"Nexus ID for lake {comid}: {nex_id_lk}")
print(f"Nexus ID for gage {gage}: {nex_id_gage}")

nex = gdf_nex.loc[gdf_nex["nex_id"].isin(nex_id_lk), :].copy()

m = lk_poly.explore(color="#20b9d0", name="Lake Polygon")
m = lk_pt.explore(m=m, color="red", name="Lake Point", marker_kwds={"radius":10})
m = gdf_fp.explore(m=m, color="#290398", name="Flowpaths")
m = gdf_nex.explore(m=m, color="#d309c3", marker_kwds={"radius": 5}, name="Nexus")
m = nex.explore(m=m, color="orange",marker_kwds={"radius":10}, name="Lake Nexus")
m = gage_pt.explore(m=m, color="green", marker_kwds={"radius":5}, name="Gage", legend=True)
m

The lakes and gage table include columns for `fp_id`, `virtual_fp_id`, `dn_nex_id`, and `dn_virtual_nex_id` so that lakes and gages are associated without requiring table joins.

In [ ]:

display(gdf_lk.head())

In [ ]:
display(gdf_gage.head())

## Every attempt will be made to maximize the NGWPC Hydrofabric such that the number of divides between 3-10 sq. km and verify that routing computational unit lengths (derived from flowpaths and flowlines) are an integer multiple of a 300 m discretization (acceptable range: 250-350 m)

The following code plots the distribution of flowpaths lengths in NHF compared to the link segment lengths generated for t-route. While the NHF geometry itself can deviate from the 250-350 meter discretization range, the links_nodes.gpkg used for t-route consistently complies.

In [ ]:
"""Plot flowpath length distribution (nhf_1.1.3.gpkg) vs link segment lengths (links_nodes.gpkg)."""


import geopandas as gpd
import numpy as np


def flowpath_lengths(nhf_path: Path, link_nodes_path: Path, domain: str):
    conn = sqlite3.connect(nhf_path)
    nhf_km = np.array([r[0] for r in conn.execute("SELECT length_km FROM flowpaths WHERE length_km > 0")])
    conn.close()

    links = gpd.read_file(link_nodes_path, layer="links")
    link_km = (links.length / 1000).values
    link_km = link_km[link_km > 0]
    del links

    datasets = [
        (nhf_km, f"{domain} NHF flowpaths"),
        (link_km, "Link segments"),
    ]
    COLORS = ["steelblue", "darkorange"]

    bins = np.logspace(-3, np.log10(max(d[0].max() for d in datasets)), 100)
    fig, ax = plt.subplots(figsize=(10, 6))

    for i, (lengths, name) in enumerate(datasets):
        clipped = lengths[lengths >= 1e-3]
        ax.hist(clipped, bins=bins, weights=np.ones(len(clipped)) / len(lengths),
                color=COLORS[i], edgecolor="none", alpha=0.6, label=name)

    ax.set_xscale("log")
    ax.set_xlabel("Segment length (km)")
    ax.set_ylabel("Fraction of segments")
    ax.set_title(f"Segment length distribution — {' vs '.join(d[1] for d in datasets)} (normalized)")
    ax.legend(loc="upper left", fontsize=8)

    for i, (lengths, name) in enumerate(datasets):
        stats_text = (
            f"{name}\n"
            f"n = {len(lengths):,}\n"
            f"total  = {np.sum(lengths):,.0f} km\n"
            f"mean   = {np.mean(lengths):.3f} km\n"
            f"median = {np.median(lengths):.3f} km\n"
            f"P1  = {np.percentile(lengths, 1):.3f} km\n"
            f"P10 = {np.percentile(lengths, 10):.3f} km\n"
            f"P90 = {np.percentile(lengths, 90):.3f} km\n"
            f"P99 = {np.percentile(lengths, 99):.3f} km"
        )
        ax.text(0.97, 0.95 - i * 0.35, stats_text, transform=ax.transAxes,
                ha="right", va="top", fontsize=7, family="monospace",
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9))

    fig.tight_layout()
    fig.savefig(f"output_nhf_{domain}_vs_links_lengths.png", dpi=200)


In [ ]:
print("AK")
flowpath_lengths(path_ak, "TODO", "AK")

print("HI")
flowpath_lengths(path_hi, "TODO", "HI")

print("PRVI")
flowpath_lengths(path_hi, "TODO", "PRVI")